# CommaSeparatedListOutputParser
> 쉼표로 구분된 항목들을 파싱하여 python 리스트로 반환<br>
> csv : 쉼표로 구분된 값 형식의 텍스트 데이터 파일

### Parser

In [2]:
from langchain_core.output_parsers.list import CommaSeparatedListOutputParser

# 콤마로 구분된 리스트 출력 파서 초기화
output_parsers = CommaSeparatedListOutputParser()

In [24]:
# 출력 형식 지침 가져오기
# model.invoke("자기소개 해줘") -> 모델이 마음대로 답함(형식이 없음)
# 출력 형식 지침 가져오기
# 모델이 일정한 형식으로 답하도록 안내하는 문자열을 생성
format_instructions = output_parsers.get_format_instructions()

format_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

## Prompt

In [ ]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 설정
prompt = PromptTemplate(
    # 주제에 대한 두 가지를 나열하라는 템플릿
    # {subject} : 나중에 들어갈 값
    # {format_instructions} : 출력 형식 지시
    template="""
    List two {subject}.
    {format_instructions}
    """,
    input_variables=["subject"], # 입력 변수로 'subject' 사용
    # 부분 변수로 형식 지침 사용
    # format_instructions : 매번 입력받지 않고 미리 고정해 둘 값
    partial_variables={"format_instructions": format_instructions},
)

In [ ]:
# 프롬프트에 들어갈 변수값 설정
prompt.input_variables

['subject']

In [8]:
# 프롬프트의 템플릿 형태 출력
print(prompt.template)


    List two {subject}.
    {format_instructions}
    


In [ ]:
# 템플릿에 값을 넣어 실제 프롬프트가 어떻게 완성되는지 확인
# 아직 모델이 호출 전이므로, LLM의 답변이 아니라 완성된 프롬프트 문자열이 반환됨
prompt.invoke({"subject":"대한민국 음식"}).text

'\n    List two 대한민국 음식.\n    Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`\n    '

## Model
### Groq API Key

In [ ]:
from dotenv import load_dotenv

# .env 파일에서 환경변수 로드
load_dotenv()

True

In [14]:
import os 

# API 키 확인
api_key = os.getenv("GROQ_API_KEY")
if api_key:
    print("GROQ API 키가 설정되었습니다.")
else:
    print("GROQ API 키가 없습니다.")

GROQ API 키가 설정되었습니다.


In [ ]:
from langchain_groq import ChatGroq

# 정확한 모델(Precise Model)
# - temperature 낮음(0.1) : 일관되고 안정적인 정확한 답변
model = ChatGroq(
    model = "openai/gpt-oss-120b",      # 모델 명
    temperature = 0.1,                  # 값이 낮을수록 안정적인 답변이 옴, 무작위성이 줄음
    model_kwargs={
        "top_p":1.0,                    # 전체 확률분포의 대답을 가져옴, 후보 제한을 거의 두지 않음
        "frequency_penalty":0.0,        # 단어나 주제에 대한 반복을 억제
        "presence_penalty":0.0          # 새로운 아이디어를 유도
    },
    max_tokens = 2000                   # 최대 생성 토큰의 제한을 2000으로 함
)

## Chain with Parser
> 사용자 입력 -> 프롬프트 구성 -> LLM 호출 -> 출력 파싱 -> 결과 반환

In [ ]:
# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain = prompt | model | output_parsers

In [ ]:
# 체인을 실행하면 모델 응답이 파서를 거쳐 리스트 형태로 반환됨
response = chain.invoke({"subject":"대한민국 지역"})

In [23]:
response

['서울특별시', '부산광역시']